# 02f | pandas Interview Questions

This lesson covers the five most common kinds of pandas questions in OAs and take-homes. Each one uses only the tools from the previous five lessons.

About 60 minutes. Try thinking about each problem on your own for 5 minutes before looking at the hint. **A few days after finishing, redo them without looking at the answers.**

## Three things to do as soon as you get a problem

1. **What does the result table look like?** What does one row represent (a user? a day? a channel?), and which columns does it have?
2. **Which tables do you need?** What key joins them, and could the join inflate the row count?
3. **How many steps?** Usually: filter → aggregate to "one row per unit" → merge → compute the metric.

In [ ]:
import numpy as np
import pandas as pd
from practice_data import load_tables

users, orders, events, employees = load_tables()
print(users.head(3))
print(orders.head(3))
print(events.head(3))

### ✏️ Exercise 1 (Write it yourself)

**Problem 1: Conversion rate by channel**

"Converted" means the user has at least one `completed` order. For each acquisition channel (`channel`), compute:

- `n_users`: the number of users;
- `n_converted`: the number of converted users;
- `rate`: the conversion rate.

The result has four columns, `channel`, `n_users`, `n_converted`, `rate`; store it in `out`.

In [ ]:
# Write your code below

<details>
<summary><b>Hint</b></summary>

First build a boolean column `converted` on `users` (whether `user_id` is among the users with completed orders, using `.isin`). Then group by `channel`: the `sum` of the boolean column is the number converted, and its `mean` is the conversion rate.

</details>

<details>
<summary><b>Answer</b></summary>

```python
buyers = orders[orders["status"] == "completed"]["user_id"].unique()
u = users.copy()
u["converted"] = u["user_id"].isin(buyers)
out = (
    u.groupby("channel")
    .agg(n_users=("user_id", "count"), n_converted=("converted", "sum"), rate=("converted", "mean"))
    .reset_index()
)
out
```

</details>

> **The key to this problem:** the denominator is "all users," so you have to start from the `users` table. If you start from `orders`, users who never ordered aren't in the table at all, and the conversion rate is always 100%. Interviewers love to see whether you fall into this trap.

### ✏️ Exercise 2 (Write it yourself)

**Problem 2: Each user's largest order**

For each user who has placed an order, find their order with the largest amount (any status). If there's a tie, take the one with the earliest date. The result has three columns, `user_id`, `order_id`, `amount`; store it in `out`.

In [ ]:
# Write your code below

<details>
<summary><b>Hint</b></summary>

Sort by `amount` descending and `order_date` ascending, then `drop_duplicates(subset=["user_id"])` to keep each user's first row. (`groupby("user_id").head(1)` also works.)

</details>

<details>
<summary><b>Answer</b></summary>

```python
out = (
    orders.sort_values(["amount", "order_date"], ascending=[False, True])
    .drop_duplicates(subset=["user_id"], keep="first")
    [["user_id", "order_id", "amount"]]
)
out
```

</details>

### ✏️ Exercise 3 (Write it yourself)

**Problem 3: The second-highest salary**

Find the **second-highest distinct salary** in `employees` (ties count as one), and store it in the variable `second`.

In [ ]:
# Write your code below

<details>
<summary><b>Hint</b></summary>

Deduplicate with `drop_duplicates()` first, sort descending, then take the second value with `.iloc[1]`. `.iloc[k]` takes the k-th item by position (starting at 0).

</details>

<details>
<summary><b>Answer</b></summary>

```python
second = employees["salary"].drop_duplicates().sort_values(ascending=False).iloc[1]
print(second)
```

</details>

### ✏️ Exercise 4 (Write it yourself)

**Problem 4: Daily active users (DAU)**

Using the records in `events` whose `event_type` is `"login"`, compute how many **distinct** users logged in each day. The result has two columns, `event_date` and `dau`, sorted by date ascending; store it in `out`.

In [ ]:
# Write your code below

<details>
<summary><b>Hint</b></summary>

The key is `nunique` (counting distinct users), not `count`. User 1 logged in twice on January 2, so `count` would overcount by one.

</details>

<details>
<summary><b>Answer</b></summary>

```python
out = (
    events[events["event_type"] == "login"]
    .groupby("event_date")["user_id"].nunique()
    .reset_index()
    .rename(columns={"user_id": "dau"})
    .sort_values("event_date")
)
out
```

</details>

### ✏️ Exercise 5 (Write it yourself)

**Problem 5: Day-1 retention**

Treat each user's **first login** date as day 0. If they also logged in on day 1 (the next day), they count as "retained on day 1." Compute the day-1 retention rate across all users and store it in `retention` (a number between 0 and 1).

In [ ]:
logins = events[events["event_type"] == "login"][["user_id", "event_date"]].drop_duplicates().copy()
logins["event_date"] = pd.to_datetime(logins["event_date"])
# Write your code below

<details>
<summary><b>Hint</b></summary>

Three steps: (1) each user's earliest login date `first_date` (`groupby().min()`); (2) merge it back into `logins` and compute which day each record falls on, `day_n`; (3) the number of distinct users with `day_n == 1`, divided by the total number of users.

</details>

<details>
<summary><b>Answer</b></summary>

```python
logins = events[events["event_type"] == "login"][["user_id", "event_date"]].drop_duplicates().copy()
logins["event_date"] = pd.to_datetime(logins["event_date"])

first = logins.groupby("user_id")["event_date"].min().reset_index().rename(columns={"event_date": "first_date"})
m = logins.merge(first, on="user_id")
m["day_n"] = (m["event_date"] - m["first_date"]).dt.days

retained_users = m[m["day_n"] == 1]["user_id"].nunique()
retention = retained_users / first["user_id"].nunique()
print(retention)
```

</details>

> **The general recipe for retention questions:** find each user's start date → merge it back into the activity table → compute "days since start" → filter by day number and count users. Day-7 retention, weekly retention, and cohort analysis all share this structure. 03f does it again in SQL.

## Rolling windows (just know they exist)

Questions like "7-day rolling average" use `rolling`. The data must be sorted by time first, with one row per day:

In [ ]:
daily = pd.DataFrame({"day": pd.date_range("2026-01-01", periods=8), "sales": [10, 12, 9, 15, 20, 18, 25, 22]})
daily["roll3"] = daily["sales"].rolling(3).mean()          # average of the last 3 days
daily["change"] = daily["sales"].diff()                    # increase over the previous day
daily["pct_change"] = daily["sales"].pct_change()          # period-over-period growth rate
daily

## pandas checkpoint

Without looking at the answers, if you can do 4 or more of the following, you're ready for SQL:

1. Filter + select columns + sort (02b);
2. Compute several statistics by group (02c);
3. Take the top N per group (02c);
4. Fill missing values with 0 after a left join, and explain why the row count can change (02d);
5. Convert a string column to dates and aggregate by month (02e);
6. Compute a conversion rate starting from `users` (Problem 1 in this lesson);
7. State the three steps of a retention question (Problem 5 in this lesson).